<a href="https://colab.research.google.com/github/mahiiipathak/ML-LAB-/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class StatQuestRNN:
    """
    A Vanilla Recurrent Neural Network built from scratch using NumPy.
    Demonstrates feedback loop unrolling, shared weight parameters,
    and Backpropagation Through Time (BPTT).
    """
    def __init__(self, input_dim=1, hidden_dim=8, output_dim=1, seed=42):
        np.random.seed(seed)

        # Shared Weights & Biases
        self.W_xh = np.random.randn(input_dim, hidden_dim) * 0.1  # Input -> Hidden (W1)
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.1  # Recurrent Hidden Loop (W2)
        self.b_h = np.zeros((1, hidden_dim))                      # Hidden Bias (b1)

        self.W_hy = np.random.randn(hidden_dim, output_dim) * 0.1  # Hidden -> Output
        self.b_y = np.zeros((1, output_dim))                      # Output Bias
        self.hidden_dim = hidden_dim

    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def forward(self, sequence):
        """
        Unrolls the RNN through time for any sequence length T.
        """
        h = np.zeros((1, self.hidden_dim))
        h_states = [h]

        # Beep Boop: Unroll across each time step t
        for x_val in sequence:
            x_t = np.array([[x_val]])
            # h_t = ReLU(x_t * W_xh + h_{t-1} * W_hh + b_h)
            h = self.relu(np.dot(x_t, self.W_xh) + np.dot(h, self.W_hh) + self.b_h)
            h_states.append(h)

        # Final prediction output from final unrolled step h_T
        y_pred = np.dot(h, self.W_hy) + self.b_y
        return y_pred[0, 0], h_states

    def train(self, X_train, y_train, epochs=2500, lr=0.05):
        """
        Trains shared parameters using Backpropagation Through Time (BPTT).
        """
        for epoch in range(epochs):
            total_loss = 0.0
            for seq, target in zip(X_train, y_train):
                # 1. Forward Pass
                y_pred, h_states = self.forward(seq)
                loss = 0.5 * (y_pred - target) ** 2
                total_loss += loss

                # 2. Output Layer Gradient
                dy = np.array([[y_pred - target]])
                dW_hy = np.dot(h_states[-1].T, dy)
                db_y = dy

                # 3. Backpropagate Through Time (BPTT)
                dh = np.dot(dy, self.W_hy.T)

                dW_xh = np.zeros_like(self.W_xh)
                dW_hh = np.zeros_like(self.W_hh)
                db_h = np.zeros_like(self.b_h)

                T = len(seq)
                for t in reversed(range(T)):
                    x_t = np.array([[seq[t]]])
                    dh_raw = dh * self.relu_derivative(h_states[t + 1])

                    # Accumulate gradients across shared parameters
                    dW_xh += np.dot(x_t.T, dh_raw)
                    dW_hh += np.dot(h_states[t].T, dh_raw)
                    db_h += dh_raw

                    # Pass gradient backward through recurrent weight (W_hh)
                    dh = np.dot(dh_raw, self.W_hh.T)

                # 4. Parameter Update (Gradient Descent)
                self.W_hy -= lr * dW_hy
                self.b_y  -= lr * db_y
                self.W_xh -= lr * dW_xh
                self.W_hh -= lr * dW_hh
                self.b_h  -= lr * db_h

In [ ]:
# Encoded StatLand Data (Low = 0.0, Medium = 0.5, High = 1.0)
X_2day = [
    [0.0, 0.0],  # Yesterday: Low, Today: Low    -> Target: Low (0.0)
    [0.0, 0.5],  # Yesterday: Low, Today: Medium -> Target: High (1.0)
    [1.0, 0.5],  # Yesterday: High, Today: Med   -> Target: Low (0.0)
    [1.0, 1.0],  # Yesterday: High, Today: High  -> Target: High (1.0)
]
y_2day = [0.0, 1.0, 0.0, 1.0]

# Initialize and Train
rnn = StatQuestRNN(seed=42)
print("Training RNN on StatLand 2-day sequences...")
rnn.train(X_2day, y_2day, epochs=2000, lr=0.05)

print("\n--- 2-DAY PREDICTIONS ---")
for seq, target in zip(X_2day, y_2day):
    pred, _ = rnn.forward(seq)
    print(f"Input: {seq} | Target: {target:.1f} | Predicted: {pred:.3f}")

print("\n--- 3-DAY UNROLLED PREDICTION ---")
# Demonstrating flexibility: Same model predicting with 3 inputs without retraining!
seq_3day = [0.0, 0.0, 0.5]  # Day -2: Low, Day -1: Low, Today: Medium
pred_3day, _ = rnn.forward(seq_3day)
print(f"Input: {seq_3day} (3 Days) | Predicted Tomorrow: {pred_3day:.3f}")

Training RNN on StatLand 2-day sequences...

--- 2-DAY PREDICTIONS ---
Input: [0.0, 0.0] | Target: 0.0 | Predicted: 0.000
Input: [0.0, 0.5] | Target: 1.0 | Predicted: 1.000
Input: [1.0, 0.5] | Target: 0.0 | Predicted: -0.000
Input: [1.0, 1.0] | Target: 1.0 | Predicted: 1.000

--- 3-DAY UNROLLED PREDICTION ---
Input: [0.0, 0.0, 0.5] (3 Days) | Predicted Tomorrow: 1.000


In [ ]:
def simulate_gradient_issues(w_2, time_steps):
    return w_2 ** time_steps

steps_list = [4, 10, 50]

print("=" * 60)
print("VANISHING / EXPLODING GRADIENT DEMONSTRATION")
print("=" * 60)

print("\n1. Exploding Gradient (W2 = 2.0):")
for T in steps_list:
    factor = simulate_gradient_issues(2.0, T)
    print(f"  Unrolled T = {T:2d} steps | Amplification Factor: {factor:.4e}")

print("\n2. Vanishing Gradient (W2 = 0.5):")
for T in steps_list:
    factor = simulate_gradient_issues(0.5, T)
    print(f"  Unrolled T = {T:2d} steps | Decay Factor: {factor:.4e}")

VANISHING / EXPLODING GRADIENT DEMONSTRATION

1. Exploding Gradient (W2 = 2.0):
  Unrolled T =  4 steps | Amplification Factor: 1.6000e+01
  Unrolled T = 10 steps | Amplification Factor: 1.0240e+03
  Unrolled T = 50 steps | Amplification Factor: 1.1259e+15

2. Vanishing Gradient (W2 = 0.5):
  Unrolled T =  4 steps | Decay Factor: 6.2500e-02
  Unrolled T = 10 steps | Decay Factor: 9.7656e-04
  Unrolled T = 50 steps | Decay Factor: 8.8818e-16
